In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent   # if notebook is inside notebooks/

if str(ROOT / Path("src/")) not in sys.path:
    sys.path.insert(0, str(ROOT / Path("src/")))

from moire.draw_lines import plot_linecut, plot_linecut_noise
from moire.io import load_field, clean_data
from moire.signal_helpers import adaptive_smooth, local_noise
from moire.extract_behaviors import extract_upturns, extract_downturns
from hampel import hampel 

OUT_NOISE = Path('output/noise_estimates_linecut_only')
IN = ROOT / Path('source_data')
FIELDS = [87, 96, 99, 103, 74, 96.2, 151, 176]
SELECT_FIELDS = [87, 96, 99, 103, 74, 96.2, 151, 176]


In [2]:

for field in SELECT_FIELDS:

    T, nu, R = load_field(field, IN) # loads initial dataset
    T, nu, R = clean_data(T, nu, R) # sorts data and removes nans

    # Creating list of linecuts 
    linecuts = []
    for i, v in enumerate(nu):
        linecuts.append({"E" : field, "nu" : v, "rho" : R[:, i]}) 

    # Proccessing each linecut: smoothing + extraction
    for linecut in linecuts:
        rho = linecut.get("rho")

        # TODO: iterative LOESS or hampel based on T window instead of points
        # Smoothing
        rho = hampel(rho).filtered_data
        rho_smoothed = adaptive_smooth(T, rho)
        linecut.update({"rho_smoothed" : rho_smoothed})

        noise = local_noise(T, rho, rho_smoothed)
        linecut.update({"local_noise" : noise})

        # Upturn & downturn extraction  
        candidates = []
        # candidates += extract_upturns_new(T, rho)
        # candidates += extract_metallic_transitions(T, rho, candidates)
        linecut.update({"candidates" : candidates})

    # Plotting and saving single linecuts 
    num_linecuts = 10
    select_idx = np.linspace(0, len(linecuts) - 1, num_linecuts, dtype="int")
    
    for i, linecut in enumerate(linecuts):
        if i in select_idx:
            print(linecut)
            plot_linecut_noise(T, linecut, save = True, OUT = OUT)
        
        

{'E': 87, 'nu': np.float64(0.83), 'rho': array([ 30.84791236,  45.25232217,  47.84732403,  48.33936683,
        44.89319544,  37.75945237,  46.02126027,  37.89075564,
        50.85438348,  34.53158769,  49.13754111,  51.44362883,
        49.4557942 ,  52.73725022,  43.64155483,  46.09390185,
        53.80737043,  56.77068497,  54.77639634,  60.01700056,
        58.25475555,  53.59454067,  74.91799565,  68.07466561,
        77.66320141,  91.00132977,  93.77266881, 108.39696486,
        96.11187375, 124.98597608, 127.81496729, 143.06990821,
       149.86052249, 155.67169819, 170.28986082, 189.08786811,
       194.83923711, 201.28962923, 213.71541973, 223.82690416,
       228.10160746, 239.81671758, 250.71074775, 260.19047619,
       268.79898031, 296.30462642, 314.36966259, 327.10447998,
       344.71237445, 351.79032692, 374.1694038 , 396.20355444]), 'rho_smoothed': array([ 38.61441456,  41.03543638,  41.96954529,  42.85285214,
        43.175037  ,  44.44003107,  45.17877815,  45.913841